In [5]:
import requests
import json
import datetime


In [6]:
url = "http://localhost:11434/api/generate"

In [7]:
import json

file_path = 'C:/Users/User/Downloads/nusrat/medrag/data/medmcqa/train.json'

# Read and parse the JSONL file
with open(file_path, 'r') as f:
    train_data = []
    for line in f:
        try:
            # Parse each line as a JSON object
            train_data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error parsing line: {line}")
            print(f"JSONDecodeError: {e}")

# Print the first few entries to verify
print(train_data[:5])

[{'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'cop': 3, 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract', 'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'choice_type': 'single'}, {'question': 'Which vitamin is supplied from only animal source:', 'exp': "Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. P 640* Vitamin B12 (Cobalamin) is synthesized solely by microorganisms.* In humans, the only source for humans is food of animal ori

In [28]:
from collections import Counter

# Count the number of questions per subject
subject_counts = Counter(question['subject_name'] for question in train_data if 'subject_name' in question)

# Print the counts
for subject, count in subject_counts.items():
    print(f"Subject: {subject}, Count: {count}")


Subject: Anatomy, Count: 14560
Subject: Biochemistry, Count: 8282
Subject: Surgery, Count: 16862
Subject: Ophthalmology, Count: 6932
Subject: Physiology, Count: 8830
Subject: Social & Preventive Medicine, Count: 11882
Subject: Gynaecology & Obstetrics, Count: 10013
Subject: Anaesthesia, Count: 3172
Subject: Psychiatry, Count: 4442
Subject: Microbiology, Count: 11314
Subject: Medicine, Count: 17887
Subject: Pharmacology, Count: 13758
Subject: Dental, Count: 8938
Subject: ENT, Count: 4919
Subject: Forensic Medicine, Count: 5900
Subject: Pediatrics, Count: 8037
Subject: Orthopaedics, Count: 2999
Subject: Radiology, Count: 4395
Subject: Pathology, Count: 14884
Subject: Skin, Count: 1771
Subject: Unknown, Count: 3045


In [4]:
sample_data = train_data[0]
print(sample_data['question'])
print(sample_data['opa'])
print(sample_data['opb'])
print(sample_data['opc'])
print(sample_data['opd'])
print(sample_data['cop'])

Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma
Hyperplasia
Hyperophy
Atrophy
Dyplasia
3


In [9]:
sample_data = train_data[0]  # Assuming train_data is already loaded

# Dynamically create the payload
payload = {
    "model": "gemma3:12b-it-qat",
    "temperature": 0,
    "system": "You are a medical doctor answering real-world exam questions. Answer only with the correct option letter (A, B, C, or D) without any additional explanation.",
    "prompt": f"""
Question: "{sample_data['question']}"

Options:
A) {sample_data['opa']}
B) {sample_data['opb']}
C) {sample_data['opc']}
D) {sample_data['opd']}
""",
    "stream": False
}

# Print the payload to verify
print(payload)

{'model': 'gemma3:12b-it-qat', 'temperature': 0, 'system': 'You are a medical doctor answering real-world exam questions. Answer only with the correct option letter (A, B, C, or D) without any additional explanation.', 'prompt': '\nQuestion: "Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma"\n\nOptions:\nA) Hyperplasia\nB) Hyperophy\nC) Atrophy\nD) Dyplasia\n', 'stream': False}


In [10]:
response = requests.post(url, json=payload)
data = response.json()

if "response" in data:
    print("Response:")
    print(data["response"])
elif "error" in data:
    print("Error:", data["error"])
else:
    print("Unexpected response format:", data)


Response:
C


In [20]:
response = requests.post(url, json=payload)
data = response.json()

if "response" in data:
    print("Response:")
    print(data["response"])
elif "error" in data:
    print("Error:", data["error"])
else:
    print("Unexpected response format:", data)

Response:
C) Atrophy


In [21]:
# Map numeric `cop` to corresponding option letter
def get_correct_option(cop):
    option_map = {1: "A", 2: "B", 3: "C", 4: "D"}
    return option_map.get(cop, None)

# Verify the model's output
def verify_model_output(sample_data, model_response):
    # Extract the correct option from `cop`
    correct_option = get_correct_option(sample_data['cop'])
    
    # Extract the model's predicted option
    predicted_option = model_response.split(")")[0].strip()  # Extract the letter before the closing parenthesis
    
    # Compare the correct option with the predicted option
    if correct_option == predicted_option:
        print("Model output is correct!")
    else:
        print("Model output is incorrect.")
        print(f"Correct Option: {correct_option}) {sample_data[f'op{correct_option.lower()}']}")
        print(f"Model Response: {model_response}")

# Example usage
sample_data = {
    "question": "Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma",
    "opa": "Hyperplasia",
    "opb": "Hyperophy",
    "opc": "Atrophy",
    "opd": "Dyplasia",
    "cop": 3  # Correct option is "C"
}

model_response = data["response"]

# Verify the model's output
verify_model_output(sample_data, model_response)

Model output is correct!


In [ ]:
response = requests.post(url, json=payload)
data = response.json()

if "response" in data:
    print("Response:")
    print(data["response"])
elif "error" in data:
    print("Error:", data["error"])
else:
    print("Unexpected response format:", data)


In [4]:
def process_single_question(question_data, url):
    """Process a single question through the model and verify the output"""
    start_time = datetime.datetime.now()
    
    # Create payload for the question
    payload = {
        "model": "gemma3:12b-it-qat",
        "temperature": 0,
        "system": "You are a medical doctor answering real-world exam questions. Answer only with the correct option letter (A, B, C, or D) without any additional explanation.",
        "prompt": f"""
Question: "{question_data['question']}"

Options:
A) {question_data['opa']}
B) {question_data['opb']}
C) {question_data['opc']}
D) {question_data['opd']}
""",
        "stream": False
    }
    
    try:
        # Make API request
        response = requests.post(url, json=payload)
        data = response.json()
        end_time = datetime.datetime.now()
        inference_time = (end_time - start_time).total_seconds()
        question_domain = question_data['subject_name']
        question_topic = question_data['topic_name']
        question_id = question_data['id']
        
        if "response" not in data:
            print(f"Error: Unexpected response format - {data}")
            return 0, None
            
        # Get model's prediction and correct answer
        model_response = data["response"]
        correct_option = get_correct_option(question_data['cop'])
        predicted_option = model_response.split(")")[0].strip()
        
        # Extract additional metrics from the response
        metrics = {
            "inference_time": inference_time,
            "total_duration": data.get("total_duration", 0),
            "load_duration": data.get("load_duration", 0),
            "prompt_eval_count": data.get("prompt_eval_count", 0),
            "prompt_eval_duration": data.get("prompt_eval_duration", 0),
            "eval_count": data.get("eval_count", 0),
            "eval_duration": data.get("eval_duration", 0)
        }
        
        # Create result dictionary
        result_dict = {
            "question_id": question_id,
            "question_domain": question_domain,
            "question_topic": question_topic,
            "question": question_data['question'],
            "correct_option": correct_option,
            "predicted_option": predicted_option,
            "is_correct": correct_option == predicted_option,
            "model_response": model_response,
            "options": {
                "A": question_data['opa'],
                "B": question_data['opb'],
                "C": question_data['opc'],
                "D": question_data['opd']
            },
            "metrics": metrics
        }
        
        # Print metrics for current question
        print(f"Inference Time: {inference_time:.2f} seconds")
        print(f"Total Duration: {metrics['total_duration']:.2f} seconds")
        print(f"Prompt Eval Count: {metrics['prompt_eval_count']}")
        
        return 1 if correct_option == predicted_option else 0, result_dict
        
    except Exception as e:
        print(f"Error processing question: {e}")
        return 0, None

def evaluate_model(train_data, url, num_samples=None, save_results=True):
    """Evaluate model performance on multiple questions and save results"""
    
    start_time = datetime.datetime.now()
    samples = train_data if num_samples is None else train_data[:num_samples]
    
    total = len(samples)
    correct = 0
    all_results = []
    total_inference_time = 0
    total_prompt_tokens = 0
    
    for i, question in enumerate(samples):
        print(f"\nProcessing question {i+1}/{total}")
        result, result_dict = process_single_question(question, url)
        correct += result
        
        if result_dict:
            all_results.append(result_dict)
            total_inference_time += result_dict["metrics"]["inference_time"]
            total_prompt_tokens += result_dict["metrics"]["prompt_eval_count"]
        
        # Print current accuracy and average metrics
        current_accuracy = (correct / (i + 1)) * 100
        avg_inference_time = total_inference_time / (i + 1)
        print(f"Current Accuracy: {current_accuracy:.2f}%")
        print(f"Average Inference Time: {avg_inference_time:.2f} seconds")
    
    end_time = datetime.datetime.now()
    total_evaluation_time = (end_time - start_time).total_seconds()
    
    # Calculate final metrics
    final_accuracy = (correct / total) * 100
    avg_inference_time = total_inference_time / total
    avg_tokens_per_question = total_prompt_tokens / total
    
    print(f"\nFinal Results:")
    print(f"Total Questions: {total}")
    print(f"Correct Answers: {correct}")
    print(f"Final Accuracy: {final_accuracy:.2f}%")
    print(f"Average Inference Time: {avg_inference_time:.2f} seconds")
    print(f"Average Tokens per Question: {avg_tokens_per_question:.2f}")
    print(f"Total Evaluation Time: {total_evaluation_time:.2f} seconds")
    
    # Save results to a JSON file
    if save_results:
        results_data = {
            "metadata": {
                "total_questions": total,
                "correct_answers": correct,
                "final_accuracy": final_accuracy,
                "model": "thewindmom/llama3-med42-8b:latest",
                "timestamp": datetime.datetime.now().isoformat(),
                "performance_metrics": {
                    "total_evaluation_time": total_evaluation_time,
                    "total_inference_time": total_inference_time,
                    "average_inference_time": avg_inference_time,
                    "total_prompt_tokens": total_prompt_tokens,
                    "average_tokens_per_question": avg_tokens_per_question
                }
            },
            "results": all_results
        }
        
        output_file = "model_evaluation_results.json"
        with open(output_file, "w") as f:
            json.dump(results_data, f, indent=2)
        print(f"\nResults saved to {output_file}")
    
    return final_accuracy, results

# Import required modules
import datetime

# Run the evaluation pipeline
url = "http://localhost:11434/api/generate"

# Evaluate questions and save results
accuracy, results = evaluate_model(train_data, url, num_samples=100, save_results=True)

NameError: name 'train_data' is not defined

### Thinking model

In [12]:
def get_correct_option(cop):
    option_map = {1: "A", 2: "B", 3: "C", 4: "D"}
    return option_map.get(cop, None)


def parse_model_response(response_text):
    """Extract thinking process and final answer from model response"""
    
    # Initialize variables
    thinking = ""
    answer = ""
    
    # Split the response into lines
    lines = response_text.strip().split('\n')
    
    # Track if we're in the thinking section
    in_thinking = False
    
    for line in lines:
        # Check for thinking section markers
        if line.strip() == "<think>":
            in_thinking = True
            continue
        elif line.strip() == "</think>":
            in_thinking = False
            continue
            
        # Collect thinking content
        if in_thinking:
            thinking += line + "\n"
        # Look for answer line
        elif line.startswith("The correct answer is"):
            answer = line.strip()
            # Extract just the letter from "The correct answer is X)"
            answer_letter = answer.split(")")[0].split()[-1]
    
    return {
        "thinking_process": thinking.strip(),
        "answer": answer_letter if answer else None
    }

def verify_model_output_with_reasoning(sample_data, model_response):
    """Verify model output including reasoning process"""
    
    # Parse the model response
    parsed_response = parse_model_response(model_response)
    thinking = parsed_response["thinking_process"]
    predicted_option = parsed_response["answer"]
    
    # Get correct option
    correct_option = get_correct_option(sample_data['cop'])
    
    # Create result dictionary
    result = {
        "is_correct": correct_option == predicted_option,
        "correct_option": correct_option,
        "predicted_option": predicted_option,
        "thinking_process": thinking,
        "question": sample_data['question'],
        "options": {
            "A": sample_data['opa'],
            "B": sample_data['opb'],
            "C": sample_data['opc'],
            "D": sample_data['opd']
        }
    }
    
    # Print results
    print(f"\nQuestion: {sample_data['question']}")
    print("\nThinking Process:")
    print(thinking)
    print(f"\nPredicted Answer: {predicted_option}")
    print(f"Correct Answer: {correct_option}")
    if result["is_correct"]:
        print("\nModel output is correct! ✓")
    else:
        print("\nModel output is incorrect. ✗")
        print(f"Correct Option: {correct_option}) {sample_data[f'op{correct_option.lower()}']}")
    
    return result

In [25]:
def process_single_question(question_data, url):
    """Process a single question through the model and verify the output"""
    start_time = datetime.datetime.now()
    
    # Create payload for the question
    payload = {
        "model": "deepseek-r1:8b",
        "temperature": 0,
        "system": """You are a medical doctor answering real-world exam questions.
        INSTRUCTIONS:
        1. First, write your thinking process between <think> and </think> tags
        2. Then, write ONLY ONE LINE with your answer in EXACTLY this format:
           'The correct answer is X)' where X is A, B, C, or D
        3. Do not write anything else after your answer.
        """,
        "prompt": f"""
Question: "{question_data['question']}"

Options:
A) {question_data['opa']}
B) {question_data['opb']}
C) {question_data['opc']}
D) {question_data['opd']}

Remember: Your response must have thinking inside <think></think> tags and end with ONLY 'The correct answer is X)'
""",
        "stream": False
    }
    
    try:
        # Make API request
        response = requests.post(url, json=payload)
        data = response.json()
        end_time = datetime.datetime.now()
        inference_time = (end_time - start_time).total_seconds()
        
        if "response" not in data:
            print(f"Error: Unexpected response format - {data}")
            return 0, None
            
        # Verify output with reasoning
        result_dict = verify_model_output_with_reasoning(question_data, data["response"])
        
        # Add metrics to result dictionary
        result_dict["metrics"] = {
            "inference_time": inference_time,
            "total_duration": data.get("total_duration", 0),
            "prompt_eval_count": data.get("prompt_eval_count", 0)
        }
        
        return 1 if result_dict["is_correct"] else 0, result_dict
        
    except Exception as e:
        print(f"Error processing question: {e}")
        return 0, None

In [26]:
process_single_question(train_data[0], url)


Question: Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma

Thinking Process:
Okay, so I've got this question about chronic urethral obstruction caused by benign prismatic hyperplasia. I'm a bit rusty on my pathology, but let me try to break it down.

First, the question is asking what happens to the kidney parenchyma in this condition. The options are hyperplasia, hyperophy, atrophy, or dyplasia. Hmm, I need to recall each term and their implications.

Benign prismatic hyperplasia refers to a benign enlargement of the prostate gland. Urethral obstruction can lead to problems with urine flow, which in turn affects the kidneys. When the urethra is blocked, the bladder may become distended, leading to increased pressure during urination. This obstruction can cause backflow of urine into the kidneys, potentially causing damage.

Now, looking at the options: 

A) Hyperplasia - that's an increase in cell number, like ben

(1,
 {'is_correct': True,
  'correct_option': 'C',
  'predicted_option': 'C',
  'thinking_process': "Okay, so I've got this question about chronic urethral obstruction caused by benign prismatic hyperplasia. I'm a bit rusty on my pathology, but let me try to break it down.\n\nFirst, the question is asking what happens to the kidney parenchyma in this condition. The options are hyperplasia, hyperophy, atrophy, or dyplasia. Hmm, I need to recall each term and their implications.\n\nBenign prismatic hyperplasia refers to a benign enlargement of the prostate gland. Urethral obstruction can lead to problems with urine flow, which in turn affects the kidneys. When the urethra is blocked, the bladder may become distended, leading to increased pressure during urination. This obstruction can cause backflow of urine into the kidneys, potentially causing damage.\n\nNow, looking at the options: \n\nA) Hyperplasia - that's an increase in cell number, like benign tumors or enlarged cells. I don't th

In [3]:
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

# Get all subject names and their counts
subjects = [q['subject_name'] for q in train_data]
total_samples = len(train_data)

# Calculate 1.1% of total samples
sample_size = int(0.011 * total_samples)

# Convert data to numpy array for sklearn
indices = np.arange(len(train_data))

# Initialize stratified split
splitter = StratifiedShuffleSplit(n_splits=1, train_size=sample_size, random_state=42)

# Get indices for the stratified sample
for sample_idx, _ in splitter.split(indices, subjects):
    stratified_sample = [train_data[i] for i in sample_idx]

# Print sample statistics
print(f"Total samples in dataset: {total_samples}")
print(f"Number of samples in 1.1% stratified sample: {len(stratified_sample)}")

# Print subject distribution in original vs sampled data
original_dist = Counter(subjects)
sample_dist = Counter(q['subject_name'] for q in stratified_sample)

print("\nSubject distribution:")
print("{:<30} {:<10} {:<10}".format("Subject", "Original", "Sampled"))
print("-" * 50)
for subject in original_dist:
    print("{:<30} {:<10} {:<10}".format(
        subject,
        original_dist[subject],
        sample_dist.get(subject, 0)
    ))

NameError: name 'train_data' is not defined